# Conexión Mongo DB

In [1]:
from pymongo import MongoClient

# 1. Conectar al contenedor de Mongo
client = MongoClient('mongodb://mongodb:27017/')

# 2. Crear (o usar) la base de datos y la colección
db = client['proyecto_bigdata'] 
coleccion = db['mercado laboral'] # Ideal el nombre del grupo ejem: 'G_Ecommerce_scraping'

print("Conexión exitosa a MongoDB")

Conexión exitosa a MongoDB


# Conectar el Scraper a la BD y guardar
Se usa el script de la semana 3

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from pymongo import MongoClient  # Librería para conectar con MongoDB
import time
import re
from urllib.parse import urljoin

# --- CONFIGURACIÓN ---
NOMBRE_GRUPO = "mercado laboral"
CATEGORIA_ACTUAL = "Programming"
URL = "https://www.getonbrd.com/jobs/programming"

options = Options()
options.add_argument("--headless=new")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=options)

# --- FUNCIONES DE APOYO ---
def limpiar_texto(txt):
    return " ".join(txt.split()).strip()

def es_valida_oferta(texto):
    low = texto.lower()
    patrones = ["programming", "developer", "engineer", "programador", "programmer", "dev", "full stack", "backend", "frontend", "software"]
    return any(p in low for p in patrones)

def es_fecha(linea):
    linea = linea.lower().strip()
    patrones = [
        r"^(ene|feb|mar|abr|may|jun|jul|ago|sep|oct|nov|dic)\s+\d{1,2}$",
        r"^(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)\s+\d{1,2}$",
    ]
    return any(re.match(p, linea) for p in patrones)

def extraer_registro(tarjeta, idx):
    lineas = [x.strip() for x in tarjeta.text.split("\n") if x.strip()]
    puesto = lineas[0] if len(lineas) > 0 else ""
    empresa = lineas[1] if len(lineas) > 1 else ""
    
    ubicacion, modalidad, salario, fecha_publicacion, url_oferta = "", "", "", "", ""

    try:
        a = tarjeta.find_element(By.TAG_NAME, "a")
        href = a.get_attribute("href")
        if href: url_oferta = urljoin(URL, href)
    except: pass

    for linea in lineas:
        low = linea.lower()
        if not ubicacion and any(x in low for x in ["santiago", "lima", "bogotá", "chile", "remote"]):
            ubicacion = linea
        if not modalidad and any(x in low for x in ["remote", "remoto", "hybrid", "híbrido"]):
            modalidad = linea
        if not salario and any(x in low for x in ["usd", "$", "mes"]):
            if any(ch.isdigit() for ch in linea): salario = linea
        if not fecha_publicacion and es_fecha(linea):
            fecha_publicacion = linea

    return {
        "puesto": puesto,
        "empresa": empresa,
        "ubicacion": ubicacion,
        "modalidad": modalidad,
        "salario": salario,
        "categoria": CATEGORIA_ACTUAL,
        "grupo": NOMBRE_GRUPO,
        "fecha_publicacion": fecha_publicacion,
        "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
        "url_oferta": url_oferta,
        "fuente": URL
    }

# --- EJECUCIÓN DEL SCRAPING ---
datos_finales = []
limite_paginas = 2

try:
    driver.get(URL)
    for nivel_pagina in range(limite_paginas):
        print(f"--- Procesando página {nivel_pagina + 1} ---")
        WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(4)

        contenedores = driver.find_elements(By.CSS_SELECTOR, "div, article, li")
        encontrados = 0
        for i, contenedor in enumerate(contenedores):
            try:
                txt = limpiar_texto(contenedor.text)
                if len(txt) < 30 or not es_valida_oferta(txt): continue
                
                registro = extraer_registro(contenedor, i)
                if registro["puesto"]:
                    datos_finales.append(registro)
                    encontrados += 1
            except: continue
        
        print(f"Registros encontrados: {encontrados}")
        try:
            siguiente = driver.find_element(By.CSS_SELECTOR, "li.next a, a.next-link")
            driver.execute_script("arguments[0].click();", siguiente)
            time.sleep(3)
        except: break

    # --- GUARDAR EN MONGODB ---
    if datos_finales:
        print(f"\nConectando a MongoDB para guardar {len(datos_finales)} registros...")
        client = MongoClient("mongodb://database:27017/")
        db = client["proyecto_bigdata"]
        coleccion = db["datos_scraping"]
        
        # BORRAR DATOS VIEJOS (Elimina los libros de Sapiens, etc.)
        coleccion.delete_many({}) 
        
        # INSERTAR NUEVOS DATOS
        coleccion.insert_many(datos_finales)
        print("¡Datos guardados exitosamente en MongoDB!")
    else:
        print("No se encontraron ofertas válidas.")

except Exception as e:
    print("Fallo en el proceso:", e)
finally:
    driver.quit()

--- Procesando página 1 ---
Registros encontrados: 365

Conectando a MongoDB para guardar 365 registros...
¡Datos guardados exitosamente en MongoDB!
